# 🏦 개인실습 — 한빛은행 시맨틱 검색 (pgvector) · 정답 모음

이 파일은 `pgvector_3_한빛은행_시맨틱_검색_개인실습.ipynb`의 정답입니다. **직접 풀어본 뒤에** 열어서 확인하세요.

- 채점 기준은 순위(top-1 문서 id)입니다 — 코사인 유사도 소수점은 근사치이니 참고만 하세요.
- 이 파일 자체로도 처음부터 실행할 수 있도록, 문제 파일의 준비 셀을 그대로 포함했습니다.

---
## ⚙️ 준비 (문제 파일과 동일 — 실행만 하세요)

In [ ]:
# 설치
# 이번 실습에 필요한 4개 패키지를 설치합니다. (-q: 설치 로그를 조용히 — quiet)
#   - sentence-transformers : 문장을 숫자 벡터(임베딩)로 바꿔주는 BGE-M3 모델을 불러올 때 사용
#   - pgvector              : PostgreSQL의 vector 타입을 파이썬 numpy 배열과 주고받게 해주는 어댑터
#   - "psycopg[binary]"     : 파이썬에서 PostgreSQL에 접속·SQL 실행을 담당하는 드라이버(라이브러리)
#   - python-dotenv         : .env 파일에 적어둔 비밀번호 등을 안전하게 읽어오는 도구
%pip install -q sentence-transformers pgvector "psycopg[binary]" python-dotenv

In [ ]:
# 멱등 준비 — 오프라인 env → 접속 → 확장·register_vector → documents 초기화 → 8문서·BGE-M3
# ※ "멱등(idempotent)"이란 이 셀을 몇 번 다시 실행해도 결과가 항상 같다는 뜻입니다.
#    (예: DROP TABLE IF EXISTS로 먼저 지우고 다시 만들기 때문에, 두 번 실행해도 에러 없이 깨끗한 상태로 시작합니다.)
import os

# 📌 오프라인 env를 모델 로딩보다 먼저! (안 하면 100초+ 지연 — rag-course 실증)
# HF_HUB_OFFLINE, TRANSFORMERS_OFFLINE 두 환경변수를 "1"로 켜두면, sentence-transformers가
# 모델을 불러올 때 인터넷에서 새 버전이 있는지 확인하러 가지 않고 곧바로 로컬 캐시(내 컴퓨터에 이미
# 받아둔 파일)를 사용합니다. 이 줄을 model 관련 import보다 반드시 "먼저" 실행해야 효과가 있습니다.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from dotenv import load_dotenv               # .env 파일 내용을 환경변수로 불러오는 함수
import numpy as np                           # 벡터(숫자 배열) 연산에 사용
import psycopg                               # PostgreSQL 접속·SQL 실행 드라이버
from pgvector.psycopg import register_vector # numpy 배열 ↔ PostgreSQL vector 타입을 자동 변환해주는 등록 함수
from sentence_transformers import SentenceTransformer  # 문장 → 벡터로 바꿔주는 임베딩 모델 클래스

# db-pg(hanbit_bank) 접속 — PostgreSQL은 비밀번호! (Redis와 다름) .env의 PGPASSWORD(load_dotenv, 하드코딩 금지)
load_dotenv()                                   # 같은 폴더의 .env → 환경변수
# .env 파일에는 보통 PGPASSWORD=이렇게생긴비밀번호 한 줄이 들어 있습니다.
# load_dotenv()를 실행하면 이 내용이 os.environ(환경변수 저장소)에 그대로 등록됩니다.
pw = os.environ["PGPASSWORD"]                   # 비밀번호는 .env 파일에 (하드코딩 금지)
# psycopg.connect(...)는 지정한 host(주소)·port(포트 번호)·dbname(데이터베이스 이름)으로
# PostgreSQL 서버에 접속을 시도하고, 성공하면 이후 SQL을 실행할 수 있는 conn(연결) 객체를 돌려줍니다.
conn = psycopg.connect(host="localhost", port=5432, dbname="hanbit_bank", user="postgres", password=pw)

# 벡터 확장 켜고(초기화용) register_vector 활성화 — 실습 1에서 직접 다시 켜봅니다
# CREATE EXTENSION IF NOT EXISTS vector : PostgreSQL에 pgvector 확장 기능(벡터 타입·거리 연산자)을
#   설치합니다. "IF NOT EXISTS"가 있어서 이미 설치돼 있으면 에러 없이 그냥 넘어갑니다.
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()                                   # commit()을 호출해야 변경 사항이 실제로 저장(확정)됩니다.
register_vector(conn)   # 📌 확장을 켠 뒤 호출 — numpy 배열을 vector로 바인딩
# register_vector(conn)을 해두면, 이후 이 conn으로 SQL을 실행할 때 numpy 배열을 자동으로
# PostgreSQL의 vector 타입으로 바꿔주고, 반대로 vector 컬럼을 읽어올 때도 numpy 배열로 돌려받습니다.

# 재실행 안전(멱등): documents만 비웁니다 (기존 customers/accounts/transactions 은행 테이블은 불간섭)
# DROP TABLE IF EXISTS documents : documents 테이블이 있으면 통째로 삭제합니다. ("IF EXISTS"라서
#   처음 실행이라 테이블이 아직 없어도 에러가 나지 않습니다.) 이 테이블은 실습 1에서 다시 만들 것입니다.
conn.execute("DROP TABLE IF EXISTS documents")
conn.commit()

# 한빛은행 FAQ/약관 8문서 (적재 순서대로 id 1~8 자동 부여)
# 아래 DOCS는 (문서 내용, 카테고리) 쌍 8개로 이루어진 파이썬 리스트입니다.
# 실습 1~4에서 이 문서들을 documents 테이블에 넣고, 문장을 벡터로 바꿔 유사도 검색을 연습합니다.
# id는 코드에 직접 적지 않고, 테이블에 넣는 "순서대로" BIGSERIAL(자동 증가)이 1부터 매겨집니다.
DOCS = [
    ("정기예금 금리는 연 3.5% 수준이며 상품별로 다르게 적용됩니다.", "예금"),         # id 1
    ("자유적금은 매달 자유롭게 납입할 수 있고 만기 시 우대 금리를 받습니다.", "예금"),   # id 2
    ("체크카드를 분실한 경우 고객센터나 모바일 앱에서 즉시 재발급을 신청할 수 있습니다.", "카드"),  # id 3
    ("신용카드 이용 한도는 고객 등급과 결제 실적에 따라 조정됩니다.", "카드"),         # id 4
    ("인터넷뱅킹 비밀번호를 5회 잘못 입력하면 계정이 잠깁니다.", "인터넷뱅킹"),        # id 5
    ("주택담보대출 한도는 소득과 담보 가치에 따라 결정됩니다.", "대출"),             # id 6
    ("예금자 보호법에 따라 원금과 이자를 합쳐 최대 5천만 원까지 보호됩니다.", "약관"),  # id 7
    ("타행 이체 수수료는 건당 500원이며 우수 등급 고객은 면제됩니다.", "이체"),        # id 8
]

# BGE-M3 임베딩 모델 (로컬 캐시·1024차원) — 로딩 약 5~10초(예시·기기마다 다름)
# SentenceTransformer("BAAI/bge-m3")는 문장을 1024개의 숫자로 이루어진 벡터로 바꿔주는 모델을
# 불러옵니다. 위에서 오프라인 env를 켜뒀기 때문에 인터넷 확인 없이 로컬 캐시에서 바로 불러옵니다.
model = SentenceTransformer("BAAI/bge-m3")

print("준비 완료 — db-pg(hanbit_bank) 접속·documents 초기화·8문서 정의·BGE-M3 로딩.")
print("문서 수:", len(DOCS))

---
## 🧪 실습 1 정답 — 벡터 확장 켜기 · 테이블 만들기 · 임베딩 맛보기

In [ ]:
# 🧪 실습 1 정답 — pgvector 확장 켜기 → documents 테이블 생성(1024차원 벡터 열) → 문장 하나를 임베딩

# 1단계: pgvector 확장을 켭니다. (이미 준비 셀에서 한 번 켰지만, 실습 목적상 여기서 직접 실행해봅니다.)
#   IF NOT EXISTS가 있어서 이미 켜져 있어도 에러 없이 넘어갑니다.
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()                        # 변경 사항을 실제로 저장(확정)합니다.
register_vector(conn)                # numpy 배열 ↔ PostgreSQL vector 타입 자동 변환을 다시 등록합니다.

# 확장이 잘 켜졌는지 버전을 조회해 눈으로 확인합니다.
# pg_extension은 PostgreSQL이 설치된 확장 목록을 담고 있는 시스템 테이블(카탈로그)입니다.
# WHERE extname='vector' 로 그중 pgvector 확장 한 줄만 골라, extversion(버전 문자열) 값을 꺼냅니다.
# .fetchone()은 결과 중 첫 번째 행 하나를 튜플로 가져오고, [0]은 그 튜플의 첫 번째 값을 꺼냅니다.
ver = conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0]
print("pgvector 확장 버전:", ver)

# 2단계: documents 테이블을 새로 만듭니다.
# 먼저 DROP TABLE IF EXISTS로 혹시 남아있을 이전 테이블을 지워, 재실행해도 항상 깨끗하게 시작합니다.
conn.execute("DROP TABLE IF EXISTS documents")
# CREATE TABLE로 4개 컬럼(열)을 가진 테이블을 만듭니다.
#   id        BIGSERIAL PRIMARY KEY : 정수 id를 1부터 자동으로 하나씩 늘려가며 채워주는 기본키(각 행의 고유 번호)
#   content   TEXT                 : 문서 원문 문장을 저장하는 텍스트 컬럼
#   embedding vector(1024)         : pgvector 확장이 제공하는 타입. 문장을 임베딩한 1024개 숫자로 이루어진 벡터를 저장
#                                     (BGE-M3 모델이 1024차원 벡터를 만들기 때문에 숫자를 1024로 지정)
#   category  TEXT                 : 문서가 속한 분류(예금/카드/대출 등) — 실습 3·4의 필터 검색에 사용
conn.execute("""
CREATE TABLE documents (
    id        BIGSERIAL PRIMARY KEY,
    content   TEXT,
    embedding vector(1024),
    category  TEXT
)
""")
conn.commit()
print("documents 테이블 생성 완료")

# 3단계: 문장 하나를 실제로 임베딩해서, 정말 1024차원 벡터가 나오는지 확인합니다.
# model.encode(문장)은 문장을 넣으면 숫자 배열(임베딩 벡터)을 돌려주는 함수입니다.
first_emb = model.encode(DOCS[0][0])
print("첫 문서 임베딩 차원:", first_emb.shape)   # (1024,) 가 나오면 테이블의 vector(1024)와 정확히 일치하는 것

---
## 🧪 실습 2 정답 — 8문서 적재 & 시맨틱 검색

In [ ]:
# 🧪 실습 2 정답 — 8문서를 임베딩해 documents에 적재 → 질문과 의미가 가까운 문서를 찾는 시맨틱 검색 함수 작성

# 1단계: DOCS 리스트에 담긴 8개 (문장, 카테고리) 쌍을 하나씩 꺼내 임베딩하고 테이블에 저장합니다.
for content, category in DOCS:
    emb = model.encode(content)                # 문장 → 1024차원 숫자 벡터로 변환
    conn.execute(
        # %s는 SQL 플레이스홀더(자리표시자)입니다. 값을 문자열로 직접 이어붙이지 않고 이렇게 넘기면
        # PostgreSQL 드라이버가 안전하게 값을 채워줍니다(SQL 인젝션 방지, 특수문자 자동 처리).
        "INSERT INTO documents (content, embedding, category) VALUES (%s, %s, %s)",
        # np.asarray(..., dtype=np.float32) : model.encode()가 돌려준 값을 pgvector가 기대하는
        # float32(32비트 실수) 타입의 numpy 배열로 통일합니다. register_vector가 이 배열을
        # 자동으로 PostgreSQL의 vector 타입으로 바꿔 넣어줍니다.
        (content, np.asarray(emb, dtype=np.float32), category)
    )
conn.commit()   # 8건 모두 넣은 뒤 한 번에 저장(확정)

# 잘 들어갔는지 개수를 세어 확인합니다. count(*)는 테이블의 전체 행 수를 세는 SQL 함수입니다.
cnt = conn.execute("SELECT count(*) FROM documents").fetchone()[0]
print("적재 문서 수:", cnt)   # 기대: 8

# 2단계: "질문 문장을 받아서 의미가 가장 비슷한 문서 k개를 찾아주는" 재사용 가능한 함수를 만듭니다.
def semantic_search(query, k=3):
    # 질문(query)도 문서와 똑같은 방식으로 임베딩해야, 같은 벡터 공간에서 "거리"를 비교할 수 있습니다.
    qe = np.asarray(model.encode(query), dtype=np.float32)
    rows = conn.execute(
        # <=>  는 pgvector가 제공하는 "코사인 거리(cosine distance)" 연산자입니다.
        #      값이 작을수록(0에 가까울수록) 두 벡터의 방향이 비슷하다 = 의미가 비슷하다는 뜻입니다.
        # 1 - (embedding <=> %s) 로 계산하면 "코사인 유사도"가 되어, 값이 클수록(1에 가까울수록) 더 비슷합니다.
        #      (거리는 작을수록 좋고, 유사도는 클수록 좋다 — 서로 반대 방향의 숫자라서 1에서 빼줍니다.)
        # ORDER BY embedding <=> %s : 거리가 가장 작은(=가장 비슷한) 순서로 정렬
        # LIMIT %s : 그중 상위 k개만 가져오기
        "SELECT id, content, category, 1 - (embedding <=> %s) AS cos_sim "
        "FROM documents ORDER BY embedding <=> %s LIMIT %s",
        (qe, qe, k)   # 같은 질문 벡터(qe)를 SELECT용과 ORDER BY용 두 번 넘겨줍니다.
    ).fetchall()      # fetchall()은 조건에 맞는 모든 행을 리스트로 가져옵니다.
    return rows

# 3단계: 실제 질문 두 개로 함수를 테스트합니다.
print("\n질의 1: 카드를 잃어버렸어요 어떻게 하죠?")
for row in semantic_search("카드를 잃어버렸어요 어떻게 하죠?"):
    print(" ", row)   # 기대 top-1: id=3(체크카드 분실 재발급), 유사도 약 0.69

print("\n질의 2: 예금자는 얼마까지 보호받나요?")
for row in semantic_search("예금자는 얼마까지 보호받나요?"):
    print(" ", row)   # 기대 top-1: id=7(예금자 보호법), 유사도 약 0.73

---
## 🧪 실습 3 정답 — ⭐필터 + 벡터 검색 결합

In [ ]:
# 🧪 실습 3 정답 ⭐ — 카테고리로 좁힌 뒤(WHERE) 벡터로 순위를 매기는 필터+벡터 결합 검색

# 질문 문장을 임베딩합니다. (실습 2의 semantic_search 함수와 같은 방식)
qe = np.asarray(model.encode("카드 한도를 늘리고 싶어요"), dtype=np.float32)

rows = conn.execute(
    # 이번에는 검색을 두 단계로 나눕니다.
    #   ① WHERE category = %s  → 먼저 "카드" 카테고리에 속한 문서로만 범위를 좁힙니다. (일반 SQL 조건 필터)
    #   ② ORDER BY embedding <=> %s → 그렇게 좁힌 문서들 안에서만 벡터 거리로 순위를 매깁니다.
    # 이렇게 하면 "카드"와 관련 없는 문서(예금·대출 등)는 애초에 후보에서 제외되므로,
    # 전체 문서를 다 비교하는 것보다 더 정확하고 더 빠르게 원하는 답을 찾을 수 있습니다.
    "SELECT id, content, category, 1 - (embedding <=> %s) AS cos_sim "
    "FROM documents WHERE category = %s "
    "ORDER BY embedding <=> %s LIMIT 3",
    (qe, "카드", qe)   # SELECT용 qe, WHERE용 "카드", ORDER BY용 qe — SQL에 나온 %s 순서와 똑같이 넘겨줍니다.
).fetchall()

print("질의 3: 카드 한도를 늘리고 싶어요 (category='카드')")
for row in rows:
    print(" ", row)   # 기대 top-1: id=4(신용카드 한도), 유사도 약 0.70

---
## 🚀 실습 4 정답 (도전·선택) — 자유적금 찾기

In [ ]:
# 🚀 추가 도전 정답 — 질의 4를 예금 카테고리로 좁혀 검색
# 실습 3과 완전히 같은 패턴(WHERE로 카테고리를 먼저 좁히고, <=>로 벡터 순위를 매기는 방식)입니다.
# 카테고리 값과 질문 문장만 바뀌었습니다 — 이 패턴이 몸에 익으면 어떤 카테고리·질문에도 그대로 응용할 수 있습니다.

qe = np.asarray(model.encode("매달 조금씩 넣는 상품 있나요?"), dtype=np.float32)
rows = conn.execute(
    "SELECT id, content, category, 1 - (embedding <=> %s) AS cos_sim "
    "FROM documents WHERE category = %s "     # 이번엔 "예금" 카테고리로 좁힙니다.
    "ORDER BY embedding <=> %s LIMIT 3",
    (qe, "예금", qe)
).fetchall()
print("질의 4: 매달 조금씩 넣는 상품 있나요? (category='예금')")
for row in rows:
    # 이 질문은 "자유적금"(id=2, 매달 자유 납입)과 "정기예금"(id=1) 둘 다 의미가 가까워서
    # 두 문서의 유사도 점수가 근소한 차이로 나옵니다 — 벡터 검색은 이렇게 "얼마나 비슷한가"를
    # 순위로 보여줄 뿐, 정답을 하나로 딱 잘라 정해주지는 않는다는 점을 보여주는 예시입니다.
    print(" ", row)   # 기대 top-1: id=2(자유적금) 약 0.52, 2위 id=1(정기예금) 약 0.50

---
## 정리

문제 파일의 확인 포인트(질의 1→id 3, 질의 2→id 7, 질의 3→id 4, 질의 4→id 2)와 비교해 보세요. 유사도 소수점은 실행 환경에 따라 조금 달라질 수 있지만, 1등 문서 id는 위와 같아야 합니다.